# Assignment 2
2025 | IN6227 Data Mining | Roshani Ayu Pranasti | G2504973A

## Install Orange Association Library

- Documentation: https://orange3-associate.readthedocs.io/en/latest/
- GitHub: https://github.com/biolab/orange3-associate/tree/master

In [6]:
!pip3 install orange3 orange3-associate

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


## Import Libraries

In [7]:
import pandas as pd
import time
import warnings
from itertools import chain, combinations # For brute-force association rules mining
from orangecontrib.associate.fpgrowth import *  # Association rules mining in Orange 3

warnings.filterwarnings('ignore')

## Dataset Definition
I have prepared 1 dataset that is Bakery Sales Dataset. More about the dataset can be read on `README.md`.
1. **Bakery Sales Dataset**: `bakery_sales.csv`

With this dataset, I will further prepare several datasets varying sizes on the number of unique items, but the number of transactions doesn’t change.

## Exploratory Data Analysis

In [8]:
# Load dataset
dataset = pd.read_csv("bakery_sales.csv")
display(dataset)

,Transaction,Item,date_time,period_day,weekday_weekend
0,1,Bread,10/30/2016 9:58,morning,weekend
1,2,Scandinavian,10/30/2016 10:05,morning,weekend
2,2,Scandinavian,10/30/2016 10:05,morning,weekend
3,3,Hot chocolate,10/30/2016 10:07,morning,weekend
4,3,Jam,10/30/2016 10:07,morning,weekend
...,...,...,...,...,...
20502,9682,Coffee,4/9/2017 14:32,afternoon,weekend
20503,9682,Tea,4/9/2017 14:32,afternoon,weekend
20504,9683,Coffee,4/9/2017 14:57,afternoon,weekend
20505,9683,Pastry,4/9/2017 14:57,afternoon,weekend


In [9]:
print("Number of attributes in dataset:", dataset.shape[1])
print("Number of data in dataset:", dataset.shape[0], "\n")
dataset.info()

Number of attributes in dataset: 5
Number of data in dataset: 20507 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20507 entries, 0 to 20506
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Transaction      20507 non-null  int64 
 1   Item             20507 non-null  object
 2   date_time        20507 non-null  object
 3   period_day       20507 non-null  object
 4   weekday_weekend  20507 non-null  object
dtypes: int64(1), object(4)
memory usage: 801.2+ KB


In [11]:
print("Number of unknown or missing values in dataset:")
dataset.isnull().sum()

Number of unknown or missing values in dataset:


Transaction        0
Item               0
date_time          0
period_day         0
weekday_weekend    0
dtype: int64

In [12]:
# Check unique values for each column
for col in dataset.columns:
    print(col, dataset[col].unique())

Transaction [   1    2    3 ... 9682 9683 9684]
Item ['Bread' 'Scandinavian' 'Hot chocolate' 'Jam' 'Cookies' 'Muffin' 'Coffee'
 'Pastry' 'Medialuna' 'Tea' 'Tartine' 'Basket' 'Mineral water'
 'Farm House' 'Fudge' 'Juice' "Ella's Kitchen Pouches" 'Victorian Sponge'
 'Frittata' 'Hearty & Seasonal' 'Soup' 'Pick and Mix Bowls' 'Smoothies'
 'Cake' 'Mighty Protein' 'Chicken sand' 'Coke' 'My-5 Fruit Shoot'
 'Focaccia' 'Sandwich' 'Alfajores' 'Eggs' 'Brownie' 'Dulce de Leche'
 'Honey' 'The BART' 'Granola' 'Fairy Doors' 'Empanadas' 'Keeping It Local'
 'Art Tray' 'Bowl Nic Pitt' 'Bread Pudding' 'Adjustment' 'Truffles'
 'Chimichurri Oil' 'Bacon' 'Spread' 'Kids biscuit' 'Siblings'
 'Caramel bites' 'Jammie Dodgers' 'Tiffin' 'Olum & polenta' 'Polenta'
 'The Nomad' 'Hack the stack' 'Bakewell' 'Lemon and coconut' 'Toast'
 'Scone' 'Crepes' 'Vegan mincepie' 'Bare Popcorn' 'Muesli' 'Crisps'
 'Pintxos' 'Gingerbread syrup' 'Panatone' 'Brioche and salami'
 'Afternoon with the baker' 'Salad' 'Chicken Stew' 'Sp

## Data Preprocessing

### Encode Transactions using One-Hot Encoding
For a simple association rules mining task, the goal is to find relationships between items within a transaction, regardless of when it happened. In this case, the time of day (`period_day`) or day of the week (`weekday_weekend`) is considered metadata about the transaction. This is why these columns were ignored when I converted the data to the required one-hot encoded format.

In [ ]:
# Use get_dummies to one-hot encode the 'Item' column
one_hot_dataset = pd.get_dummies(dataset["Item"])

# Combine it with the 'Transaction' column
one_hot_dataset = pd.concat([dataset["Transaction"], one_hot_dataset], axis=1)
display(one_hot_dataset)

# Group by transaction and sum the one-hot encoded columns
# This counts the occurrences of each item in each transaction
encoded_dataset = one_hot_dataset.groupby('Transaction').sum()
display(encoded_dataset)

# Convert counts to binary (0 or 1)
encoded_dataset = encoded_dataset.map(lambda x: 1 if x > 0 else 0)
display(encoded_dataset)

,Transaction,Adjustment,Afternoon with the baker,Alfajores,Argentina Night,Art Tray,Bacon,Baguette,Bakewell,Bare Popcorn,...,The BART,The Nomad,Tiffin,Toast,Truffles,Tshirt,Valentine's card,Vegan Feast,Vegan mincepie,Victorian Sponge
0,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,3,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,3,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20502,9682,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20503,9682,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20504,9683,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20505,9683,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


,Adjustment,Afternoon with the baker,Alfajores,Argentina Night,Art Tray,Bacon,Baguette,Bakewell,Bare Popcorn,Basket,...,The BART,The Nomad,Tiffin,Toast,Truffles,Tshirt,Valentine's card,Vegan Feast,Vegan mincepie,Victorian Sponge
Transaction,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9680,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9681,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
9682,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


,Adjustment,Afternoon with the baker,Alfajores,Argentina Night,Art Tray,Bacon,Baguette,Bakewell,Bare Popcorn,Basket,...,The BART,The Nomad,Tiffin,Toast,Truffles,Tshirt,Valentine's card,Vegan Feast,Vegan mincepie,Victorian Sponge
Transaction,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9680,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9681,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
9682,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Vary Dataset in Different Sizes based on Number of Unique Items
I will vary sizes on the number of unique items, but keeping the number of transactions doesn’t change

In [31]:
# Define constants to vary dataset
n_transactions = 127
n_unique_items_list = [7, 8, 9, 10, 11, 12]

for n_unique_items in n_unique_items_list:
    print("Unique Items:", n_unique_items)
    coba = encoded_dataset.iloc(one_hot_dataset.iloc[:n_transactions, 2:n_unique_items+2])
    # display(coba)

Unique Items: 7


TypeError: unhashable type: 'DataFrame'

## Orange 3 Association Rules Mining

Based on its GitHub codebase, Orange 3 uses the FP-Growth (Frequent Pattern Growth) algorithm for its association rules mining.
- Orange 3 associate FP-growth: https://github.com/biolab/orange3-associate/blob/master/orangecontrib/associate/fpgrowth.py

In [ ]:
min_support = 0.01
min_confidence = 0.4

In [ ]:
# Create a mapping from column index to item name
mapping = {i: item_name for i, item_name in enumerate(encoded_dataset.columns)}

# Find frequent itemsets
start_orange_association_rules = time.time()
itemsets = dict(frequent_itemsets(encoded_dataset.values, min_support=min_support))

# Generate association rules from the frequent itemsets
rules = list(association_rules(itemsets, min_confidence=min_confidence))
end_orange_association_rules = time.time()

# Present processed rules as results
results = []
for antecedent, consequent, support, confidence in rules:
    antecedent_list = [mapping[item] for item in antecedent]
    consequent_list = [mapping[item] for item in consequent]

    results.append({
        "Antecedent": ", ".join(antecedent_list),
        "Consequent": ", ".join(consequent_list),
        "Support": support,
        "Confidence": confidence
    })

# Display results
orange_association_rules_time = end_orange_association_rules - start_orange_association_rules
print("Dataset 1 orange 3 association rules mining time:", orange_association_rules_time)
results_df = pd.DataFrame(results)
print("Results:")
display(results_df)

# Sort the DataFrame by "Confidence" to see the most interesting rules first
results_df = results_df.sort_values(by="Confidence", ascending=False)
print("Top 5 with highest confidence:")
display(results_df.head(5))

## Brute-Force Association Rules Mining

In [ ]:
# Define brute-force algorithm function
def get_subsets(items):
    """
    Generates all non-empty subsets from a list of items.
    Example: get_subsets(['a', 'b']) -> {'a'}, {'b'}, {'a', 'b'}
    """
    return chain.from_iterable(combinations(items, r) for r in range(1, len(items) + 1))

def brute_force_association_rules(transactions, min_support, min_confidence):
    """
    Generates association rules using a brute force approach.

    Args:
        transactions (list of sets): The transaction database.
        min_support (float): The minimum support threshold.
        min_confidence (float): The minimum confidence threshold.

    Returns:
        list: A list of tuples, where each tuple represents a rule
              (antecedent, consequent, support, confidence).
    """
    unique_items = sorted(list(set(item for transaction in transactions for item in transaction)))
    all_itemsets = [frozenset(subset) for subset in get_subsets(unique_items)]
    
    num_transactions = len(transactions)
    
    # Step 1: Calculate Support for All Itemsets
    itemset_supports = {}
    for itemset in all_itemsets:
        count = 0
        for transaction in transactions:
            if itemset.issubset(transaction):
                count += 1
        support = count / num_transactions
        if support >= min_support:
            itemset_supports[itemset] = support

    final_rules = []

    # Step 2: Generate and Test All Rules
    for itemset, support in itemset_supports.items():
        if len(itemset) > 1:
            # Generate all possible rules from this itemset
            for antecedent in (frozenset(subset) for subset in get_subsets(itemset) if len(subset) < len(itemset)):
                consequent = itemset - antecedent
                
                # Check if the antecedent exists in our support dictionary
                if antecedent in itemset_supports:
                    antecedent_support = itemset_supports[antecedent]
                    confidence = support / antecedent_support
                    
                    if confidence >= min_confidence:
                        final_rules.append((antecedent, consequent, support, confidence))

    return final_rules

In [ ]:
# Convert data into a list of sets
def convert_into_list_of_sets(data):
    transactions = []
    for index, row in data.iterrows():
        # For each row, get the column names where the value is 1
        itemset = set(row.index[row == 1])
        transactions.append(itemset)

    return transactions

# Run brute_force_association
def run_brute_force_association_rules(data, min_support, min_confidence):
    transactions = convert_into_list_of_sets(data)

    start_brute_force_association_rules = time.time()
    rules = brute_force_association_rules(transactions, min_support=min_support, min_confidence=min_confidence)
    end_brute_force_association_rules = time.time()

    brute_force_association_rules_time = end_brute_force_association_rules - start_brute_force_association_rules

    return rules


In [ ]:
# rules = run_brute_force_association_rules(encoded_dataset.head(100), min_support=min_support, min_confidence=min_confidence)

# Display results
# display(rules)